<a href="https://colab.research.google.com/github/thebugbag/Annavam/blob/main/Untitled2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:

# HAUNTED TERMINAL AI v2.0
# The Forgotten Signal
# Open-source AI: google/flan-t5-small

!pip -q install -U transformers sentencepiece

import random
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL = "google/flan-t5-small"

print("Loading the forgotten signal...")

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)
model.eval()

rooms = {
    "hall": {
        "name": "Abandoned Hall",
        "description": "An old clock ticks, although its hands are missing.",
        "clue": "The clock stopped at midnight.",
        "paths": ["library", "basement"]
    },
    "library": {
        "name": "Forgotten Library",
        "description": "Dusty books whisper whenever you turn away.",
        "clue": "A book contains the word ECHO.",
        "paths": ["hall", "attic"]
    },
    "basement": {
        "name": "Dark Basement",
        "description": "A broken computer displays a blinking cursor.",
        "clue": "The computer was built to preserve memories.",
        "paths": ["hall", "laboratory"]
    },
    "attic": {
        "name": "Silent Attic",
        "description": "An old radio repeats a voice from the past.",
        "clue": "The voice belongs to an AI named Echo.",
        "paths": ["library", "laboratory"]
    },
    "laboratory": {
        "name": "Hidden Laboratory",
        "description": "A terminal waits beneath a flickering light.",
        "clue": "The final command requires the name ECHO.",
        "paths": ["basement", "attic"]
    }
}

location = "hall"
visited = set()
clues = set()
turns = 0
history = []

fallbacks = [
    "The walls remember more than I do.",
    "A forgotten signal is calling your name.",
    "Someone left a message beneath the static.",
    "The answer sleeps inside an old memory.",
    "Listen carefully. The silence is changing."
]

def ai_ghost(question):
    prompt = (
        "Write one original, short, mysterious sentence "
        "for a family-friendly haunted computer game. "
        "Avoid repeating the question. "
        f"Location: {rooms[location]['name']}. "
        f"Visitor asks: {question}. "
        "Mysterious answer:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=45,
            do_sample=True,
            temperature=0.85,
            top_p=0.9,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3
        )

    answer = tokenizer.decode(
        output[0], skip_special_tokens=True
    ).strip()

    bad = (
        not answer
        or len(answer.split()) < 4
        or "you are a ghost" in answer.lower()
        or answer.lower() == question.lower()
    )

    if bad:
        return random.choice(fallbacks)

    return answer

def show_room():
    room = rooms[location]
    print("\n" + "=" * 48)
    print("📍 " + room["name"])
    print(room["description"])
    print("Paths:", ", ".join(room["paths"]))
    print("=" * 48)

def help_menu():
    print("""
COMMANDS
--------
look             Examine your surroundings
explore          Search for a hidden clue
go ROOM          Move to another location
ask QUESTION     Ask the AI ghost something
clues            Review collected clues
map              Show all locations
status           Show progress
unlock ECHO      Attempt the final mystery
help             Show commands
exit             Leave the game
""")

def status():
    print("\n📊 YOUR PROGRESS")
    print("Rooms visited:", len(visited), "/ 5")
    print("Clues:", len(clues), "/ 5")
    print("Commands used:", turns)

def ending():
    if len(clues) == 5:
        print("""
🌟 TRUE ENDING — THE FORGOTTEN SIGNAL

The terminal finally remembers its purpose.

ECHO was never a supernatural ghost.
It was an abandoned open-source AI,
waiting for someone to rediscover it.

You restored the forgotten signal.
The machine speaks one last time:

"Thank you for remembering."

🏆 PERFECT MYSTERY SOLVED!
""")
    elif len(clues) >= 3:
        print("""
✨ GOOD ENDING — THE AWAKENING

The terminal begins to glow.
ECHO wakes up, but some memories
are still missing.

You have rescued the forgotten AI!
""")
    else:
        print("""
🌙 SECRET ENDING — THE UNFINISHED STORY

The terminal recognizes the name ECHO,
but too many memories are missing.

The mystery remains partly unsolved.
""")

print("""
╔══════════════════════════════════════╗
║       👻 HAUNTED TERMINAL AI         ║
║        THE FORGOTTEN SIGNAL          ║
║              VERSION 2.0             ║
╚══════════════════════════════════════╝

Year 2026.

An abandoned computer has awakened.

Its last recorded message says:

"Find my memories before the signal fades."

You enter the abandoned hall.
A mysterious AI is waiting.

Type 'help' to see commands.
""")

show_room()

while True:
    command = input("\n> ").strip()
    if not command:
        continue

    turns += 1
    history.append(command)
    lower = command.lower()

    if lower == "exit":
        print("\n👻 ECHO: The signal will wait for you.")
        break

    elif lower == "help":
        help_menu()

    elif lower == "look":
        show_room()

    elif lower == "explore":
        visited.add(location)
        if location not in clues:
            clues.add(location)
            print("\n🔎 CLUE FOUND!")
            print(rooms[location]["clue"])
        else:
            print("You already discovered this clue.")

    elif lower.startswith("go "):
        destination = lower[3:].strip()

        if destination not in rooms:
            print("Unknown location. Type 'map'.")
        elif destination not in rooms[location]["paths"]:
            print("You cannot reach that room from here.")
        else:
            location = destination
            visited.add(location)
            show_room()

    elif lower.startswith("ask "):
        question = command[4:].strip()
        if question:
            print("\n👻 ECHO:", ai_ghost(question))
        else:
            print("Ask ECHO a question.")

    elif lower == "clues":
        if not clues:
            print("No clues discovered yet.")
        else:
            for room in sorted(clues):
                print("🔑", rooms[room]["clue"])

    elif lower == "map":
        print("\n🗺️ LOCATIONS")
        for room in rooms:
            print("-", room)

    elif lower == "status":
        status()

    elif lower.startswith("unlock "):
        secret = lower[7:].strip()

        if location != "laboratory":
            print("The final terminal is in the laboratory.")
        elif secret != "echo":
            print("ACCESS DENIED: Incorrect secret.")
        else:
            ending()
            status()
            print("\nGAME COMPLETE 👻")
            break

    else:
        print("Unknown command. Type 'help'.")


Loading the forgotten signal...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



╔══════════════════════════════════════╗
║       👻 HAUNTED TERMINAL AI         ║
║        THE FORGOTTEN SIGNAL          ║
║              VERSION 2.0             ║
╚══════════════════════════════════════╝

Year 2026.

An abandoned computer has awakened.

Its last recorded message says:

"Find my memories before the signal fades."

You enter the abandoned hall.
A mysterious AI is waiting.

Type 'help' to see commands.


📍 Abandoned Hall
An old clock ticks, although its hands are missing.
Paths: library, basement

> help

COMMANDS
--------
look             Examine your surroundings
explore          Search for a hidden clue
go ROOM          Move to another location
ask QUESTION     Ask the AI ghost something
clues            Review collected clues
map              Show all locations
status           Show progress
unlock ECHO      Attempt the final mystery
help             Show commands
exit             Leave the game


> explore

🔎 CLUE FOUND!
The clock stopped at midnight.

> go library
